# xLSTM[7:1] multi-task training — EgoExo-Fitness on Colab T4

Trains the shared-encoder xLSTM with classification + quality regression + comment generation (Approach B, frozen Flan-T5-small + soft prefix). Inference returns `{quality, comment, guidance}`.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**.
2. Push your local `Finess-coach-capstone-1` repo to a GitHub repo *or* zip+upload to Drive.
3. Make sure your EgoExo CLIP features are on Drive at `MyDrive/egoexo_fitness_full/features_open/visual/EgoExo_Fitness_CLIP_Vid_Feat_w_Rotate/...`.

**Wall-clock estimate on T4** (≈1k clips, 40 epochs, batch=32, comment head ON): ~25–35 min.

## 1. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free,driver_version --format=csv
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 2. Mount Drive (CLIP features + checkpoint output)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive'
DATA_ROOT  = f'{DRIVE_ROOT}/egoexo_fitness_full'              # ← edit if your Drive layout differs
OUT_ROOT   = f'{DRIVE_ROOT}/xlstm_egoexo_multitask_T4'        # checkpoints will land here
os.makedirs(OUT_ROOT, exist_ok=True)
assert os.path.isdir(DATA_ROOT), f'EgoExo data not found at {DATA_ROOT} — adjust DATA_ROOT'
print('data:', DATA_ROOT)
print('out :', OUT_ROOT)

MessageError: [dfs_ephemeral] Credentials propagation unsuccessful

## 3. Pull the project code

Pick **one** of the two options below.

In [ ]:
# --- Option A: clone from GitHub (replace with your repo URL) ---
%cd /content
!git clone https://github.com/<your-user>/Finess-coach-capstone-1.git
%cd /content/Finess-coach-capstone-1

# --- Option B: copy a zip from Drive instead (uncomment if no GitHub) ---
# !cp '/content/drive/MyDrive/Finess-coach-capstone-1.zip' /content/repo.zip
# !unzip -q /content/repo.zip -d /content/
# %cd /content/Finess-coach-capstone-1

!pwd && ls -1 | head

## 4. Install dependencies

In [ ]:
# Colab already has torch + numpy + sklearn. Add the comment-head deps.
!pip -q install 'transformers>=4.30' 'sentencepiece>=0.1.99' huggingface_hub
# install the project itself in editable mode so `from fitness_coach...` works
!pip -q install -e .

## 5. Symlink the dataset into the expected paths

Avoids copying GBs of CLIP features by reading them straight from Drive.

In [ ]:
import os
os.makedirs('notebooks/data', exist_ok=True)
if not os.path.exists('notebooks/data/egoexo_fitness_full'):
    os.symlink(DATA_ROOT, 'notebooks/data/egoexo_fitness_full')
!ls -la notebooks/data/egoexo_fitness_full | head
!ls notebooks/data/egoexo_fitness_full/features_open/visual/EgoExo_Fitness_CLIP_Vid_Feat_w_Rotate | head -5
!ls notebooks/data/egoexo_fitness_full/raw_annotations

## 6. Build the index CSV (one-time, only if you don't already have one)

Skip this cell if `results/egoexo_fitness_index_split.csv` is already in the cloned repo.

In [ ]:
import os
if not os.path.isfile('results/egoexo_fitness_index_split.csv'):
    !python build_egoexo_fitness_index.py \
      --raw-annotations notebooks/data/egoexo_fitness_full/raw_annotations \
      --output results/egoexo_fitness_index.csv
    !python split_exercise_index.py \
      --input results/egoexo_fitness_index.csv \
      --output results/egoexo_fitness_index_split.csv \
      --val-frac 0.15 --test-frac 0.15 --seed 42
    !ln -sf egoexo_fitness_index_split.csv results/egoexo_index.csv
!head -2 results/egoexo_fitness_index_split.csv && echo '---' && wc -l results/egoexo_fitness_index_split.csv

## 7. Train

Settings tuned for T4 (16 GB VRAM): batch 32, hidden 256, xLSTM[7:1], attention pool + multi-task fusion ON, frozen Flan-T5-small comment head, AdamW with 10 % linear warmup + cosine decay.

In [ ]:
import shlex, subprocess, time

cmd = f'''python train_xlstm_egoexo_multitask.py \
  --index-csv results/egoexo_fitness_index_split.csv \
  --feature-mode clip \
  --clip-features-root notebooks/data/egoexo_fitness_full/features_open/visual \
  --clip-view ego_l --clip-max-frames 300 --clip-subsample-stride 3 \
  --hidden 256 --layers 8 --num-heads 4 --conv-kernel-size 4 --projection-factor 1.333 \
  --block-pattern mmmmmmms \
  --use-attention-pool --use-fusion --fusion-dim 128 \
  --dropout 0.15 \
  --optimizer adamw --lr 3e-4 --weight-decay 1e-4 \
  --warmup-frac 0.1 --min-lr-ratio 0.05 --grad-clip 1.5 \
  --cls-weight 0.9 --reg-weight 0.1 --error-weight 0.0 --comment-weight 0.1 \
  --comment-head --lm-name google/flan-t5-small --n-prefix 16 \
  --balanced-class-weights --standardize \
  --epochs 40 --batch-size 32 --num-workers 2 \
  --eval-test \
  --output-dir {shlex.quote(OUT_ROOT)}'''

print(cmd)
t0 = time.time()
rc = subprocess.call(cmd, shell=True)
print(f'\n[exit={rc}] elapsed={(time.time()-t0)/60:.1f} min')

## 8. Quick training-curve plot

In [ ]:
import json, matplotlib.pyplot as plt
with open(f'{OUT_ROOT}/training_history.json') as f:
    hist = json.load(f)
ep = [h['epoch'] for h in hist]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot(ep, [h['train_loss'] for h in hist], label='train loss')
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('loss'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(ep, [h['val_accuracy'] for h in hist], label='val acc')
ax[1].plot(ep, [h['val_f1_macro']  for h in hist], label='val f1-macro')
ax[1].plot(ep, [1 - h['val_mae']    for h in hist], label='1 − val MAE (quality)')
ax[1].set_xlabel('epoch'); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 9. Inference demo — `{quality, comment, guidance}`

In [ ]:
import torch, json
from fitness_coach.models.xlstm_model import xLSTMExerciseClassifier, CommentGenerationHead
from fitness_coach.datasets.egoexo_xlstm_dataset import (
    EgoExoXLSTMDataset, ERROR_TAGS, egoexo_collate_fn,
)
from torch.utils.data import DataLoader

ckpt_path = f'{OUT_ROOT}/xlstm_egoexo_multitask_best.pt'
ckpt = torch.load(ckpt_path, map_location='cuda', weights_only=False)
device = torch.device('cuda')

model = xLSTMExerciseClassifier(
    input_size=ckpt['input_size'], hidden_size=ckpt['hidden'], num_layers=ckpt['layers'],
    num_classes=len(ckpt['classes']), dropout=ckpt['dropout'], num_heads=ckpt['num_heads'],
    conv_kernel_size=ckpt['conv_kernel_size'], projection_factor=ckpt['projection_factor'],
    num_error_tags=len(ckpt['error_tags']) if 'error_tags' in ckpt and ckpt['weights'].get('err',0)>0 else 0,
    block_pattern=ckpt.get('block_pattern'), use_attention_pool=ckpt.get('use_attention_pool', False),
    use_fusion=ckpt.get('use_fusion', False), fusion_dim=ckpt.get('fusion_dim', 128),
).to(device)
model.load_state_dict(ckpt['model'])
model.set_guidance_table(ckpt.get('guidance_table', {}), ckpt.get('idx_to_class', {}))
model.eval()

comment_head = None
if ckpt.get('comment_head', {}).get('enabled'):
    ch = ckpt['comment_head']
    comment_head = CommentGenerationHead(
        encoder_dim=ckpt['hidden'], error_tags=ckpt['error_tags'],
        model_name=ch['lm_name'], n_prefix_tokens=ch['n_prefix'],
        max_target_len=ch['max_target_len'], max_prompt_len=ch['max_prompt_len'],
    ).to(device)
    comment_head.prefix_proj.load_state_dict(ckpt['comment_prefix_proj'])
    comment_head.eval()

# Pull a few val samples
from pathlib import Path
val = EgoExoXLSTMDataset(
    Path('results/egoexo_fitness_index_split.csv'),
    ckpt['class_to_idx'], 'val', feature_mode='clip',
    clip_features_root=Path('notebooks/data/egoexo_fitness_full/features_open/visual'),
    clip_view='ego_l', clip_max_frames=300, clip_subsample_stride=3,
)
loader = DataLoader(val, batch_size=4, shuffle=True, collate_fn=egoexo_collate_fn)
xb, y_cls, y_q, y_err, gold_comments, gold_classes = next(iter(loader))
results = model.infer(xb.to(device), comment_head=comment_head)
for r, gold_c, gold_q in zip(results, gold_comments, y_q.tolist()):
    print(json.dumps({**r, 'gold_quality': gold_q, 'gold_comment': gold_c}, indent=2))
    print('-' * 80)

## 10. (Optional) Resume / continue from a checkpoint

If Colab disconnects mid-training, just re-run the training cell — the script will overwrite from scratch. To **continue** instead, add `--resume <ckpt path>` (only if you've extended the training script with that flag; not built in by default).